# 10 KV Cache and Inference

## Problem

为什么大模型逐 token 生成时需要 KV cache？如果没有 cache，到底重复计算了什么？cache 真正省下来的到底是算力、显存，还是时间？

## Dependencies

建议先完成 `04_self_attention_from_scratch.ipynb`、`05_multi_head_attention.ipynb`、`09_build_a_basic_transformer_block.ipynb`。


## Goals

- 理解自回归生成为什么会产生大量重复计算
- 理解为什么通常缓存 K 和 V，而不是缓存完整 attention 输出
- 能解释有无 cache 时，计算路径到底差在哪里
- 能把 KV cache 的收益和代价都说清楚

## Scope and Approach

这一节不把 KV cache 写成一句“推理时缓存 K/V 提速”就结束。我们会先把自回归生成的时间线拆开，再看没有 cache 时到底重复了什么，有 cache 后具体省掉了哪一段。


## 自回归生成为什么会重复劳动

假设模型正在逐 token 生成一句话。生成第 1 个 token 后，再生成第 2 个 token；生成第 2 个 token 后，再生成第 3 个 token。问题在于，每次新 token 到来时，attention 都需要看历史上下文。

如果我们完全不做缓存，那么在第 `t` 步时，模型会：

- 重新把前 `t` 个 token 再走一遍投影
- 重新得到前 `t` 个位置的 K 和 V
- 再让最后一个位置用它的 Q 去和这些历史 K 做匹配

这里最浪费的地方是：历史 token 的 K/V 明明上一步已经算过了，但下一步又被从头算一遍。


In [ ]:
import numpy as np

np.random.seed(21)
np.set_printoptions(precision=3, suppress=True)

hidden_dim = 4
W_q = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_k = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_v = np.random.randn(hidden_dim, hidden_dim) * 0.2

# 用 3 个 token 构造一个很小的生成场景。
sequence = [
    np.array([1.0, 0.0, 0.5, 0.2]),
    np.array([0.8, 0.1, 0.3, 0.0]),
    np.array([0.1, 1.0, 0.4, 0.6]),
]


## 没有 cache 时，重复计算在哪里

我们先写一个最小过程，不做任何缓存。注意这里不是为了做高性能实现，而是为了把“重复劳动”打印出来。


In [ ]:
def no_cache_generate(seq):
    total_kv_computes = 0

    for t in range(1, len(seq) + 1):
        # 第 t 步时，我们把前缀全部重新堆起来。
        prefix = np.stack(seq[:t])

        # 历史所有位置的 K/V 都重新算一遍。
        K = prefix @ W_k
        V = prefix @ W_v

        # 当前最后一个位置的 Q 也要算。
        Q = prefix[-1:] @ W_q

        total_kv_computes += len(prefix)
        print(f't={t}: prefix_len={len(prefix)}, K.shape={K.shape}, V.shape={V.shape}, Q.shape={Q.shape}')

    return total_kv_computes

print('无 cache 的 K/V 计算次数 =', no_cache_generate(sequence))


## 为什么缓存的是 K 和 V

在 decoder-only 生成里，第 `t` 步新来的是“最后一个 token”。这意味着：

- 新 token 的 `Q` 必须新算，因为它代表“当前这个位置在找什么”
- 历史 token 的 `K/V` 通常不变，因为历史内容没有变

所以最自然的思路就是：

- 历史 K/V 算过一次就存起来
- 下一步只算新 token 的 K/V，再 append 到 cache 里
- 当前 Q 去和 cache 里的全部 K 做匹配

这就是 KV cache 的核心。


In [ ]:
def with_cache_generate(seq):
    k_cache = []
    v_cache = []
    total_new_kv_computes = 0

    for t, token in enumerate(seq, start=1):
        # 这里只需要对“新 token”算一次 K/V。
        k_new = token @ W_k
        v_new = token @ W_v
        q_new = token @ W_q

        k_cache.append(k_new)
        v_cache.append(v_new)
        total_new_kv_computes += 1

        K = np.stack(k_cache)
        V = np.stack(v_cache)

        print(f't={t}: cache_len={len(k_cache)}, K.shape={K.shape}, V.shape={V.shape}, q_new.shape={q_new.shape}')

    return total_new_kv_computes

print('有 cache 的新 K/V 计算次数 =', with_cache_generate(sequence))


## cache 真正省下来的是什么

KV cache 最直接省下来的，是**历史 token 的重复投影计算**。随着序列越来越长，这种节省会越来越明显。

但它不是白送的，因为它也带来代价：

- 你要把每一层、每个历史位置的 K/V 都存起来
- 上下文越长，cache 越大
- 推理瓶颈会越来越转向显存占用和带宽搬运

所以 KV cache 不是“省显存”的方案，而是“用显存换计算时间”的方案。


## 为什么这会引出后面的 MQA、GQA、MLA

当上下文变长、层数变深、head 变多时，KV cache 会迅速变成推理系统里的大头之一。于是后面的很多设计，本质上都在回答一个问题：

**能不能在尽量不损失太多效果的前提下，把 K/V 存得更省一点？**

这就是为什么后面要引出：

- MQA：减少 K/V head 的数量
- GQA：在效果和 cache 成本之间做折中
- MLA：从状态表示层面进一步压缩缓存成本

## Common mistakes

- 把 KV cache 理解成缓存完整 attention 输出。常见做法缓存的是历史 K 和 V。
- 以为有了 cache 就没有代价。它节省重复计算，但会带来显存占用和带宽压力。
- 不区分训练和推理。KV cache 主要是推理阶段的关键工程结构。
- 以为 cache 只是“代码优化”。实际上它会深刻影响后面整个注意力结构的工程设计。

## Checkpoints

- 用自己的话解释没有 cache 时，重复计算到底发生在哪里。
- 解释为什么通常缓存 K/V，而不是缓存 Q。
- 说明为什么长上下文场景下，KV cache 会变成主要瓶颈之一。
- 回答：KV cache 主要是在省算力，还是在省显存？

## Summary

KV cache 的意义在于：历史 token 的 K/V 算过一次之后，不必在后续每一步生成时反复重算。它本质上是在用显存和带宽，换掉重复投影带来的时间开销。理解这一点以后，后面再看 MQA、GQA、MLA，就会非常顺。

## Next Module

下一节进入 MHA、MQA、GQA。那一节会真正把“怎么省 K/V cache”这条工程主线拆开来看。
